In [1]:
import pyrosetta
import numpy as np

from benchmark import full_bpti_benchmark
from energy_calculation import calculate_and_compare_energies
from misc import init_basic_params, default_qaoa_params, BasicParams, QAOAParams
from rotamer_extraction import extract_top_n_rotamers, TrackedResidue
from h_ising_creation import build_ising_hamiltonian, extract_and_reduce_tensors
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [11]:
# Pyrosetta Relevant Code
benchmark_pose = full_bpti_benchmark()
residue_library, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic, global_offset = extract_and_reduce_tensors(residue_library, ig)
basic_params: BasicParams = init_basic_params(h_linear)
qaoa_params: QAOAParams = default_qaoa_params()

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_linear, J_quadratic)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, basic_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function, qaoa_params)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
valid_conformations = validate_conformations(probabilities, basic_params)

# Calculate both the quantum and pyrosetta energies for comparison
calculate_and_compare_energies(valid_conformations,
                               h_linear, J_quadratic, global_offset,
                               benchmark_pose, scorefxn, residue_library,
                               basic_params)


Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
Moltenres ID: 1, SeqPos ID: 20
Moltenres ID: 2, SeqPos ID: 21
Moltenres ID: 3, SeqPos ID: 22
Moltenres ID: 4, SeqPos ID: 23
Moltenres ID: 5, SeqPos ID: 24
initializing generator_params
Reduced Hamiltonian built: 14 Qubits, 55 Pauli strings.
Commencing QAOA Optimization [p=2]...
Epoch   0 | Cost: -47.0949
Epoch  10 | Cost: -47.0950
Epoch  20 | Cost: -47.0951
Epoch  30 | Cost: -47.0951
Epoch  40 | Cost: -47.0951
Epoch  50 | Cost: -47.0951
Epoch  60 | Cost: -47.0951
Epoch  70 | Cost: -47.0951
Epoch  80 | Cost: -47.0951
Epoch  90 | Cost: -47.0951
Epoch 100 | Cost: -47.0951
Epoch 110 | Cost: -47.0951
Epoch 120 | Cost: -47.0951
Epoch 130 | Cost: -47.0951
Epoch 140 | Cost: -47.0951
Optimization converged.
{20: 0, 21: 2, 23: 6, 24: 10}
Valid to Non-Val

In [ ]:
temp_go = global_offset

In [ ]:
h_linear

In [ ]:
J_quadratic

In [9]:
import importlib, energy_calculation, validation
importlib.reload(validation)
importlib.reload(energy_calculation)

from energy_calculation import calculate_and_compare_energies
calculate_and_compare_energies(valid_conformations, h_linear, J_quadratic, 0, benchmark_pose, scorefxn, residue_library, basic_params)

==================== ENERGY OPERATIONS ====================
Calculating Quantum Energies for all 100 conformations
Calculating Pyrosetta for all 100 conformations 
Comparing both energy types for all 100 conformations 
The difference was too large to ignore for conformation 14.699632960404088: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0], probability=np.float64(0.9999999999736209), quantum_energy=np.float64(-25.43668570369482), biological_energy=np.float64(-40.13631866409891), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x113e1a2f0>)
The difference was too large to ignore for conformation 14.699631829007572: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0], probability=np.float64(2.998657387264138e-12), quantum_energy=np.float64(-25.435701869428158), biological_energy=np.float64(-40.13533369843573), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x1134d8430>)
The difference was too large to ignore for conformation 14.699632450836631: C

In [16]:
import importlib, energy_calculation, validation
importlib.reload(validation)
importlib.reload(energy_calculation)

from energy_calculation import calculate_and_compare_energies
calculate_and_compare_energies(valid_conformations, h_linear, J_quadratic, global_offset, benchmark_pose, scorefxn, residue_library, basic_params)

==================== ENERGY OPERATIONS ====================
Calculating Quantum Energies for all 100 conformations
Calculating Pyrosetta for all 100 conformations 
Comparing both energy types for all 100 conformations 
The difference was too large to ignore for conformation 14.699632960404088: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], probability=np.float64(0.9999999999736218), quantum_energy=np.float64(-25.43668570369482), biological_energy=np.float64(-40.13631866409891), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x11458ac30>)
The difference was too large to ignore for conformation 14.699631829007572: Conformation(bitstring=[1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], probability=np.float64(2.9986511060311126e-12), quantum_energy=np.float64(-25.435701869428158), biological_energy=np.float64(-40.13533369843573), pose=<pyrosetta.rosetta.core.pose.Pose object at 0x107a40b70>)
The difference was too large to ignore for conformation 14.699632450836631: Confor

In [7]:
error_reduced = [np.float64(48.2082289401788), np.float64(48.20822780878228), np.float64(48.208187172904374), np.float64(48.20822900850408), np.float64(48.209747403977545), np.float64(48.20818625548229), np.float64(48.27781879250436), np.float64(48.192365427215535), np.float64(48.200900020276734), np.float64(48.20811899153513), np.float64(48.2081189545482), np.float64(48.20822787710756), np.float64(48.208187241229595), np.float64(48.209746272581), np.float64(48.209705517493774), np.float64(48.209747472302766), np.float64(48.20818632380757), np.float64(48.20970460007166), np.float64(48.2778176611079), np.float64(48.27777702522994), np.float64(48.27781886082958), np.float64(48.277776107807796), np.float64(48.19236429581899), np.float64(48.19232365994104), np.float64(48.19236549554081), np.float64(48.19388389101431), np.float64(48.19232274251898), np.float64(48.2619552795411), np.float64(48.200898888880275), np.float64(48.20085813379302), np.float64(48.20090008860201), np.float64(48.200857216370935), np.float64(48.18503650731347), np.float64(48.208117860138614), np.float64(48.208077224260705), np.float64(48.209637455333876), np.float64(48.20807630683868), np.float64(48.20811782315168), np.float64(48.20807718727377), np.float64(48.209637418346944), np.float64(48.208076269851745), np.float64(48.277708843860694), np.float64(48.27770880687376), np.float64(48.192255480326395), np.float64(48.19225544333946), np.float64(48.20079007163312), np.float64(48.20079003464619), np.float64(48.20974634090628), np.float64(48.20970558581905), np.float64(48.209704668396995), np.float64(48.277817729433124), np.float64(48.27777709355516), np.float64(48.277776176133074), np.float64(48.19236436414427), np.float64(48.19232372826626), np.float64(48.193882759617765), np.float64(48.19384200453047), np.float64(48.19388395933953), np.float64(48.1923228108442), np.float64(48.19384108710841), np.float64(48.26195414814461), np.float64(48.2619135122666), np.float64(48.26195534786632), np.float64(48.26191259484449), np.float64(48.20089895720555), np.float64(48.2008582021183), np.float64(48.20085728469621), np.float64(48.18503537591698), np.float64(48.18499462082974), np.float64(48.18503657563869), np.float64(48.18499370340757), np.float64(48.20963632393733), np.float64(48.209595568850105), np.float64(48.20959465142805), np.float64(48.2096362869504), np.float64(48.20959553186317), np.float64(48.209594614441116), np.float64(48.277707712464235), np.float64(48.27766707658627), np.float64(48.277666159164184), np.float64(48.2777076754773), np.float64(48.277667039599336), np.float64(48.27766612217725), np.float64(48.19225434892985), np.float64(48.1922137130519), np.float64(48.1937739441251), np.float64(48.192212795629786), np.float64(48.19225431194292), np.float64(48.192213676064966), np.float64(48.193773907138166), np.float64(48.19221275864285), np.float64(48.26184533265189), np.float64(48.261845295664955), np.float64(48.20078894023666), np.float64(48.20074818514941), np.float64(48.200747267727266), np.float64(48.20078890324973), np.float64(48.200748148162475), np.float64(48.20074723074033), np.float64(48.184926560424316)]

np.mean(error_reduced), np.std(error_reduced)

(np.float64(48.21719144874189), np.float64(0.031482728737122496))

In [10]:
error_normal = [np.float64(14.699632960404088), np.float64(14.699631829007572), np.float64(14.699632450836631), np.float64(14.699633028729366), np.float64(14.694622256749199), np.float64(14.699631533414546), np.float64(14.694344762839734), np.float64(14.683775206342915), np.float64(14.67138652124018), np.float64(14.699632578438042), np.float64(14.699632541451109), np.float64(14.69963189733285), np.float64(14.69963251916191), np.float64(14.69462112535274), np.float64(14.694621627972452), np.float64(14.694622325074477), np.float64(14.699631601739881), np.float64(14.694620710550424), np.float64(14.694343631443274), np.float64(14.694344253272277), np.float64(14.694344831165012), np.float64(14.694343335850249), np.float64(14.683774074946399), np.float64(14.683774696775458), np.float64(14.683775274668193), np.float64(14.678764502688054), np.float64(14.683773779353345), np.float64(14.67848700877856), np.float64(14.67138538984372), np.float64(14.671385892463434), np.float64(14.671386589565458), np.float64(14.671384975041406), np.float64(14.655528767179007), np.float64(14.699631447041526), np.float64(14.699632068870613), np.float64(14.694621874783152), np.float64(14.699631151448585), np.float64(14.699631410054593), np.float64(14.69963203188368), np.float64(14.69462183779622), np.float64(14.699631114461653), np.float64(14.694344380873673), np.float64(14.69434434388674), np.float64(14.683774826131398), np.float64(14.683774789144465), np.float64(14.67138613927419), np.float64(14.671386102287258), np.float64(14.69462119367796), np.float64(14.69462169629773), np.float64(14.694620778875702), np.float64(14.694343699768496), np.float64(14.694344321597612), np.float64(14.69434340417547), np.float64(14.683774143271677), np.float64(14.68377476510068), np.float64(14.678763371291595), np.float64(14.678763873911294), np.float64(14.678764571013332), np.float64(14.683773847678566), np.float64(14.678762956489237), np.float64(14.678485877382101), np.float64(14.678486499211076), np.float64(14.678487077103782), np.float64(14.678485581788962), np.float64(14.671385458168999), np.float64(14.671385960788768), np.float64(14.671385043366683), np.float64(14.655527635782548), np.float64(14.655528138402232), np.float64(14.655528835504228), np.float64(14.655527220980119), np.float64(14.694620743386693), np.float64(14.69462124600642), np.float64(14.694620328584392), np.float64(14.69462070639976), np.float64(14.694621209019488), np.float64(14.69462029159746), np.float64(14.694343249477214), np.float64(14.694343871306302), np.float64(14.694342953884217), np.float64(14.694343212490281), np.float64(14.694343834319369), np.float64(14.694342916897284), np.float64(14.683773694734882), np.float64(14.683774316563856), np.float64(14.678764122476537), np.float64(14.683773399141828), np.float64(14.683773657747949), np.float64(14.683774279576923), np.float64(14.678764085489604), np.float64(14.683773362154895), np.float64(14.678486628566958), np.float64(14.678486591580025), np.float64(14.671385007877731), np.float64(14.671385510497458), np.float64(14.671384593075373), np.float64(14.671384970890799), np.float64(14.671385473510526), np.float64(14.67138455608844), np.float64(14.655528386967461)]

np.mean(error_normal), np.std(error_normal)

(np.float64(14.685340311495485), np.float64(0.0122728968924223))

In [13]:
residue_library

{20: TrackedResidue(moltenres_idx=1, seqpos=20, rotamers=[TrackedRotamer(one_body_energy=-7.102415561676025, original_pyrosetta_index=1, residue=<pyrosetta.rosetta.core.conformation.Residue object at 0x113f956f0>), TrackedRotamer(one_body_energy=-5.602204322814941, original_pyrosetta_index=2, residue=<pyrosetta.rosetta.core.conformation.Residue object at 0x113f97270>)]),
 21: TrackedResidue(moltenres_idx=2, seqpos=21, rotamers=[TrackedRotamer(one_body_energy=-12.81937026977539, original_pyrosetta_index=1, residue=<pyrosetta.rosetta.core.conformation.Residue object at 0x113f96bb0>), TrackedRotamer(one_body_energy=-12.815899848937988, original_pyrosetta_index=2, residue=<pyrosetta.rosetta.core.conformation.Residue object at 0x113f96d70>), TrackedRotamer(one_body_energy=-7.861598014831543, original_pyrosetta_index=5, residue=<pyrosetta.rosetta.core.conformation.Residue object at 0x113f97d30>), TrackedRotamer(one_body_energy=-7.856298923492432, original_pyrosetta_index=6, residue=<pyrosett

In [18]:
energies = [{"quantum_energy": conf.quantum_energy, "biological_energy": conf.biological_energy, "probability": conf.probability} for conf in valid_conformations]

energies.sort(key=lambda conf: conf['probability'], reverse=True)
for i, conf in enumerate(energies):
    conf['probs_idx'] = i

energies.sort(key=lambda conf: conf['quantum_energy'])
for i, conf in enumerate(energies):
    conf['quant_idx'] = i

energies.sort(key=lambda conf: conf['biological_energy'])
for i, conf in enumerate(energies):
    conf['bio_idx'] = i
energies

rank_match = [abs(conf['quant_idx'] - conf['bio_idx']) for idx, conf in enumerate(energies)]
rank_match_results =  (np.mean(rank_match[:10]), np.mean(rank_match), np.mean(rank_match[-10:]))
prob_rank_match = [abs(conf['probs_idx'] - conf['bio_idx']) for idx, conf in enumerate(energies)]
results = (np.mean(prob_rank_match[:10]), np.mean(prob_rank_match), np.mean(prob_rank_match[-10:]))

print("Quantum-Bio Rank match means", rank_match_results)
print("Prob Rank match means", results)

Quantum-Bio Rank match means (np.float64(0.0), np.float64(0.0), np.float64(0.0))
Prob Rank match means (np.float64(15.1), np.float64(15.88), np.float64(11.7))


In [ ]:
prob_rank_match = [abs(conf['probs_idx'] - conf['bio_idx']) for idx, conf in enumerate(energies)]
results = (np.mean(prob_rank_match[:10]), np.mean(prob_rank_match), np.mean(prob_rank_match[-10:]))
print("Rank match means", results)

In [ ]:
from validation import Conformation
from rotamer_extraction import TrackedRotamer
import itertools

one_body = []
two_body = []

def evaluate_two_energies_alt(pose, valid_conformations: list[Conformation], scorefxn, residue_library: dict[int, TrackedResidue], params, ig, rot_sets, h_linear, J_quadratic):
    for conformation in valid_conformations:

        bitstring = conformation.bitstring
        bio_energy, _ = evaluate_singular_pyrosetta_energy_alt(pose, bitstring, scorefxn, residue_library, params)
        # conformation.biological_energy = bio_energy

        quant_energy = evaluate_quantum_energy_alt(bitstring, residue_library, params, ig, rot_sets, h_linear, J_quadratic)
        # conformation.quantum_energy = quant_energy
        print("Bio:", bio_energy, conformation.biological_energy, bio_energy - conformation.biological_energy)
        print("Qua:", quant_energy, conformation.quantum_energy, quant_energy - conformation.quantum_energy)
        print("Diff:", bio_energy-quant_energy)
        print()

    # print("==============ONE BODY ENERGIES==============")
    # print(one_body)
    #
    # print("==============TWO BODY ENERGIES==============")
    # print(two_body)


def evaluate_singular_pyrosetta_energy_alt(pose, bitstrings, scorefxn, residue_library: dict[int, TrackedResidue], params):
    test_pose = pose.clone()

    seq_positions = params["seq_positions"]
    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]

    for seq in seq_positions:
        base_wire = wire_offsets[seq]
        num_rots = rotamer_counts[seq]

        residue_bits = bitstrings[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        res_entry: TrackedResidue = residue_library[seq]
        rotamer_entry = res_entry.rotamers[local_rotamer_idx]

        test_pose.replace_residue(seq, rotamer_entry.residue, False)

    bio_energy = scorefxn(test_pose)
    return bio_energy, test_pose

def evaluate_quantum_energy_alt(bitstring, residue_library: dict[int, TrackedResidue], params, ig, rot_sets, h_linear, J_quadratic):
    seq_positions = params["seq_positions"]
    seq_to_molten = {rot_sets.moltenres_2_resid(m): m for m in range(1, rot_sets.nmoltenres() + 1)}
    print(seq_positions)

    wire_offsets = params["wire_offsets"]
    rotamer_counts = params["rotamer_counts"]


    def get_picked_rotamer_idx(seq_idx):
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        rotamer_seq_entry = residue_library[seq_idx]
        rotamer_entry = rotamer_seq_entry.rotamers[local_rotamer_idx]

        return rotamer_entry.original_pyrosetta_index

    one_body_energies = 0
    for seq_idx in seq_positions:
        base_wire = wire_offsets[seq_idx]
        num_rots = rotamer_counts[seq_idx]

        residue_bits = bitstring[base_wire : base_wire + num_rots]
        local_rotamer_idx = residue_bits.index(1)

        single_energy = residue_library[seq_idx].rotamers[local_rotamer_idx].one_body_energy
        alt_single_energy = h_linear[seq_idx][local_rotamer_idx]

        assert single_energy == alt_single_energy, f"Energies do not match {seq_idx} - {local_rotamer_idx}: {single_energy} != {alt_single_energy}"

        one_body_energies += single_energy

    alt_one_body_energies = 0

    for seq, energies in h_linear.items():
            base_wire = wire_offsets[seq]
            # print(base_wire)
            for rot, e_val in energies.items():
                # print("\t", rot, "=>", base_wire+rot)
                if bitstring[base_wire + rot] == 1:
                    # print("\t\t", "Found item in bitstring", rot, e_val)
                    alt_one_body_energies += e_val
    one_body.append((one_body_energies, alt_one_body_energies, one_body_energies == alt_one_body_energies))

    two_body_energies = 0
    for seq_i, seq_j in itertools.combinations(seq_positions, 2):
        molten_i = seq_to_molten[seq_i]
        molten_j = seq_to_molten[seq_j]

        if not ig.get_edge_exists(molten_i, molten_j): continue
        edge = ig.find_edge(molten_i, molten_j)

        rot_index_i = get_picked_rotamer_idx(seq_i)
        rot_index_j = get_picked_rotamer_idx(seq_j)

        pair_energy = edge.get_two_body_energy(rot_index_i, rot_index_j)
        two_body_energies += pair_energy

    alt_two_body_energies = 0
    for (seq_i, seq_j), interactions in J_quadratic.items():
            for (rot_i, rot_j), e_val in interactions.items():
                k = wire_offsets[seq_i] + rot_i
                l = wire_offsets[seq_j] + rot_j

                if bitstring[k] == 1 and bitstring[l] == 1:
                    alt_two_body_energies += e_val
    two_body.append((two_body_energies, alt_two_body_energies, two_body_energies == alt_two_body_energies))

    return one_body_energies + two_body_energies
evaluate_two_energies_alt(benchmark_pose, valid_conformations, scorefxn, residue_library, basic_params, ig, rot_sets, h_linear, J_quadratic)

In [ ]:
from pyrosetta.rosetta.core.scoring import automorphic_rmsd

def get_alphafold_baseline_indices(pose, rot_sets, active_start=20, active_end=24):
    """
    Projects the continuous AlphaFold pose side-chains onto the discrete
    Dunbrack rotamer sets to find the closest matching InteractionGraph states.
    """
    baseline_ig_states = {} # Format: {Node_ID: State_ID}

    for pose_res_idx in range(active_start, active_end + 1):

        # 1. Map Pose Index to Graph Node Index (MoltenRes)
        # If the residue is fixed, this returns 0. For active residues, it returns 1, 2, 3...
        node_id = rot_sets.resid_2_moltenres(pose_res_idx)

        if node_id == 0:
            continue # Failsafe: Should not happen if TaskFactory is correct

        # 2. Retrieve the specific subset of Dunbrack rotamers for this node
        rot_set = rot_sets.rotamer_set_for_residue(pose_res_idx)
        num_states = rot_set.num_rotamers()

        # 3. Geometrically project: Find the rotamer with the minimum RMSD to AlphaFold
        best_state_id = 1
        min_rmsd = float('inf')
        original_residue = pose.residue(pose_res_idx)

        for state_id in range(1, num_states + 1):
            # Extract the specific discrete rotamer geometry
            rotamer_conformation = rot_set.rotamer(state_id)

            # Calculate structural deviation (Heavy-atom RMSD accounts for symmetry)
            rmsd = automorphic_rmsd(rotamer_conformation, original_residue, False)

            if rmsd < min_rmsd:
                min_rmsd = rmsd
                best_state_id = state_id

        # 4. Record the mapping
        baseline_ig_states[node_id] = best_state_id

        # Optional: Log the RMSD to prove in your thesis how close the discrete mapping is
        print(f"Pose Res {pose_res_idx} -> Node {node_id} | AF matched to State {best_state_id} (RMSD: {min_rmsd:.3f} Å)")

    return baseline_ig_states
baseline_ig_states = get_alphafold_baseline_indices(benchmark_pose, rot_sets)

In [ ]:
valid_conformations.sort(key= lambda conf: conf.biological_energy)
# [0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0]
valid_conformations

In [ ]:
{key: len(val.rotamers) for key, val in residue_library.items()}

In [ ]:
baseline_ig_states

In [ ]:
# 1. Get the global AlphaFold energy
total_af_energy = scorefxn(benchmark_pose)

# 2. Calculate the discrete graph energy for the matched states
ig_baseline_energy = 0.0

# Add 1-body terms
for node_i, state_a in baseline_ig_states.items():
    ig_baseline_energy += ig.get_one_body_energy_for_node_state(node_i, state_a)
    print([rot.one_body_energy for rot in residue_library[node_i+19].rotamers])
    print(ig_baseline_energy, "\n==============================\n")

# Add 2-body terms
for node_i, state_a in baseline_ig_states.items():
    for node_j, state_b in baseline_ig_states.items():
        if not node_j > node_i: continue
        if not ig.get_edge_exists(node_i, node_j): continue

        edge = ig.find_edge(node_i, node_j)
        ig_baseline_energy += edge.get_two_body_energy(state_a, state_b)

# 3. Isolate the Constant
invariant_background_energy = total_af_energy - ig_baseline_energy

In [ ]:
total_af_energy

In [ ]:
ig_baseline_energy

In [ ]:
invariant_background_energy

In [ ]:
basic_params